# A/B Testing Toolkit Exploration

This notebook follows the same workflow as `main.py`, with each module
explored interactively so the data, assumptions, and plots remain visible.

## Simulation

Simulation gives us a known ground truth, letting us check whether an
analysis can recover a deliberately planted treatment effect.

In [ ]:
import matplotlib.pyplot as plt

from config import (
    ALPHA,
    BASE_CONVERSION_RATE,
    MINIMUM_DETECTABLE_EFFECT,
    POWER,
    RANDOM_SEED,
    SAMPLE_SIZE_PER_GROUP,
)
from src.simulation import generate_experiment_data

simulation_df = generate_experiment_data(
    SAMPLE_SIZE_PER_GROUP, SAMPLE_SIZE_PER_GROUP,
    BASE_CONVERSION_RATE, MINIMUM_DETECTABLE_EFFECT, RANDOM_SEED
)
simulation_df.groupby("group")["converted"].mean()

## Statistical Tests

A proportions z-test evaluates a binary conversion metric. A t-test is
better suited to continuous metrics such as revenue per user, while a
chi-square test checks independence in a count table.

In [ ]:
from src.stats import run_chi_square, run_proportions_ztest, run_ttest

control = simulation_df[simulation_df["group"] == "control"]
treatment = simulation_df[simulation_df["group"] == "treatment"]
z_result = run_proportions_ztest(
    int(control["converted"].sum()), len(control),
    int(treatment["converted"].sum()), len(treatment)
)
chi_result = run_chi_square([[
    int(control["converted"].sum()),
    len(control) - int(control["converted"].sum())
], [
    int(treatment["converted"].sum()),
    len(treatment) - int(treatment["converted"].sum())
]])
ttest_result = run_ttest([1.0, 1.2, 0.9], [1.1, 1.4, 1.0])
z_result, chi_result, ttest_result

## Power Analysis

Power analysis is a pre-experiment decision: it tells us how much data
we need to reliably detect an effect that matters to the business.

In [ ]:
from src.power import (
    calculate_sample_size,
    plot_power_curve,
    plot_sample_size_vs_mde,
)

planned_n = calculate_sample_size(
    BASE_CONVERSION_RATE, MINIMUM_DETECTABLE_EFFECT, ALPHA, POWER
)
plot_power_curve(
    BASE_CONVERSION_RATE, MINIMUM_DETECTABLE_EFFECT, ALPHA,
    range(500, 6001, 500), "../plots/notebook_power_curve.png"
)
plot_sample_size_vs_mde(
    BASE_CONVERSION_RATE, ALPHA, POWER,
    [0.005, 0.01, 0.02, 0.03, 0.04],
    "../plots/notebook_sample_size_vs_mde.png"
)
from IPython.display import Image, display
display(Image(filename="../plots/notebook_power_curve.png"))
display(Image(filename="../plots/notebook_sample_size_vs_mde.png"))
plt.show()

## Pitfalls

Optional stopping and many simultaneous metrics can turn ordinary noise
into apparently exciting findings. Segment-level analysis also guards
against Simpson's paradox.

In [ ]:
from src.pitfalls import (
    generate_simpsons_paradox_data,
    plot_peeking_fpr,
    simulate_multiple_comparisons,
    simulate_peeking,
)

peeking_results = simulate_peeking(500, 3000, 0.10,
                                  [500, 1000, 1500, 2000, 3000],
                                  ALPHA, RANDOM_SEED)
plot_peeking_fpr(peeking_results, ALPHA,
                 "../plots/notebook_peeking_fpr.png")
display(Image(filename="../plots/notebook_peeking_fpr.png"))
multiple_results = simulate_multiple_comparisons(20, 1000, ALPHA, RANDOM_SEED)
simpsons_data = generate_simpsons_paradox_data(RANDOM_SEED)
multiple_results, simpsons_data

## Stakeholder Reporting

A statistical result is useful only when a decision-maker can understand
the size of the change, the uncertainty, and the recommended next step.

In [ ]:
from src.reporting import summarize_power_analysis, summarize_test_result

summary = summarize_test_result(
    z_result, "Conversion rate",
    control["converted"].mean(), treatment["converted"].mean()
)
power_summary = summarize_power_analysis(
    planned_n, MINIMUM_DETECTABLE_EFFECT, ALPHA, POWER
)
logger = __import__("logging").getLogger("notebook")
logger.info(summary)
logger.info(power_summary)

## What I learned

- Sample size is a design decision that must happen before looking at results.
- Statistical significance and practical business impact answer different questions.
- Repeated peeking makes a nominal 5% threshold less trustworthy.
- Testing many metrics increases the chance of finding noise that looks real.
- Segment context can reverse an aggregate conclusion, so averages need investigation.